# LFW 03-00. Fallback-free open-set 결과 검증

`02_step1_compression_characterization.ipynb`이 생성한 open-set 검색 결과를
읽기 전용으로 검증합니다. PCA/PQ 검색을 다시 수행하거나 origin exact fallback으로
결과를 대체하지 않습니다.

검증 항목:

- active LFW run과 result manifest의 extraction lineage
- 결과 파일 SHA-256
- `origin_fallback_used=False`
- frozen-origin/recalibrated-compressed 정책별 DIR, FPIR 재계산 일치
- 명시적 embedding exclusion 수


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


MODE = "real"
DATA_FRACTION = 1.0
SEED = 42
MODEL_NAME = "arcface"
EXECUTE_STAGE = True


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun 프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.experiments.scope import ExperimentScope
from research.runtime import resolve_active_run
from research.runtime.hashing import sha256_file

EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
RUN_ROOT = PROJECT_ROOT / "runs" / "lfw"
RESULT_ROOT = PROJECT_ROOT / "results" / "step1" / "lfw"
SCOPE_TAG = f"{MODE}_p{DATA_FRACTION:.4f}_s{SEED}_{MODEL_NAME}"


In [ ]:
RUN_DIR = resolve_active_run(RUN_ROOT, allow_completed=True)
RUN_MANIFEST = json.loads(
    (RUN_DIR / "run_manifest.json").read_text(encoding="utf-8")
)
RUN_ID = str(RUN_MANIFEST["run_id"])
RESULT_DIR = RESULT_ROOT / RUN_ID / SCOPE_TAG
RESULT_MANIFEST_PATH = RESULT_DIR / "result_manifest.json"

if not RESULT_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        "Step 1 결과가 없습니다. 먼저 "
        "notebooks/lfw/02_compression/02_step1_compression_characterization.ipynb"
        "을 실행하십시오."
    )

RESULT_MANIFEST = json.loads(
    RESULT_MANIFEST_PATH.read_text(encoding="utf-8")
)
if RESULT_MANIFEST.get("scope") != EXPERIMENT_SCOPE.as_dict():
    raise ValueError(
        f"result scope mismatch: expected={EXPERIMENT_SCOPE.as_dict()}, "
        f"actual={RESULT_MANIFEST.get('scope')}"
    )
if RESULT_MANIFEST.get("source", {}).get("extraction_run_id") != RUN_ID:
    raise ValueError("active run과 Step 1 extraction lineage가 다릅니다.")

verified_files = {}
for name, expected in RESULT_MANIFEST.get("files", {}).items():
    path = RESULT_DIR / name
    if not path.is_file():
        raise FileNotFoundError(path)
    actual_sha256 = sha256_file(path)
    if actual_sha256 != expected.get("sha256"):
        raise ValueError(f"result hash mismatch: {name}")
    verified_files[name] = {
        "bytes": path.stat().st_size,
        "sha256": actual_sha256,
    }

preflight = {
    "execute_stage": EXECUTE_STAGE,
    "run_id": RUN_ID,
    "result_dir": str(RESULT_DIR),
    "scope": EXPERIMENT_SCOPE.as_dict(),
    "verified_files": sorted(verified_files),
    "input_coverage": RESULT_MANIFEST.get("input_coverage"),
}
display(pd.Series(preflight, name="value").to_frame())


In [ ]:
def strict_bool(values: pd.Series, *, label: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(values):
        return values.astype(bool)
    normalized = values.astype(str).str.strip().str.lower()
    mapped = normalized.map({"true": True, "false": False})
    if mapped.isna().any():
        raise ValueError(f"{label} contains non-boolean values.")
    return mapped.astype(bool)


result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    retrieval = pd.read_csv(RESULT_DIR / "retrieval_comparison.csv")
    recorded_summary = pd.read_csv(RESULT_DIR / "compression_summary.csv")
    exclusions = pd.read_csv(RESULT_DIR / "embedding_exclusions.csv")

    required = {
        "compression_family",
        "compression_profile",
        "threshold_policy",
        "is_mated",
        "compressed_accepted",
        "compressed_rank1_correct",
        "agreement_with_origin",
        "threshold_crossing",
        "origin_fallback_used",
    }
    missing = sorted(required.difference(retrieval.columns))
    if missing:
        raise ValueError(f"retrieval result columns missing: {missing}")
    if strict_bool(
        retrieval["origin_fallback_used"], label="origin_fallback_used"
    ).any():
        raise RuntimeError("fallback-free Step 1 결과에 origin fallback 행이 있습니다.")

    records = []
    group_columns = [
        "compression_family",
        "compression_profile",
        "threshold_policy",
    ]
    for keys, group in retrieval.groupby(group_columns, sort=True):
        family, profile, policy = keys
        mated = strict_bool(group["is_mated"], label="is_mated")
        accepted = strict_bool(
            group["compressed_accepted"], label="compressed_accepted"
        )
        correct = strict_bool(
            group["compressed_rank1_correct"], label="compressed_rank1_correct"
        )
        records.append(
            {
                "compression_family": family,
                "compression_profile": profile,
                "threshold_policy": policy,
                "query_count": int(len(group)),
                "dir_rank1_recomputed": (
                    float((accepted & correct & mated).sum() / mated.sum())
                    if mated.any()
                    else np.nan
                ),
                "fpir_recomputed": (
                    float((accepted & ~mated).sum() / (~mated).sum())
                    if (~mated).any()
                    else np.nan
                ),
                "agreement_with_origin": float(
                    strict_bool(
                        group["agreement_with_origin"],
                        label="agreement_with_origin",
                    ).mean()
                ),
                "threshold_crossing_rate": float(
                    strict_bool(
                        group["threshold_crossing"],
                        label="threshold_crossing",
                    ).mean()
                ),
            }
        )
    recomputed = pd.DataFrame.from_records(records)
    comparison = recomputed.merge(
        recorded_summary[
            group_columns + ["dir_rank1", "fpir"]
        ],
        on=group_columns,
        how="left",
        validate="one_to_one",
    )
    comparison["dir_abs_diff"] = (
        comparison["dir_rank1_recomputed"] - comparison["dir_rank1"]
    ).abs()
    comparison["fpir_abs_diff"] = (
        comparison["fpir_recomputed"] - comparison["fpir"]
    ).abs()
    if comparison[["dir_abs_diff", "fpir_abs_diff"]].max().max() > 1e-12:
        raise RuntimeError("저장된 open-set summary와 재계산 결과가 다릅니다.")

    expected_exclusions = int(
        RESULT_MANIFEST.get("input_coverage", {}).get("excluded_rows", 0)
    )
    if len(exclusions) != expected_exclusions:
        raise ValueError("embedding exclusion CSV와 result manifest가 다릅니다.")

    display(comparison)
    result = {
        "status": "verified_read_only",
        "run_id": RUN_ID,
        "profiles": int(comparison["compression_profile"].nunique()),
        "policy_rows": int(len(comparison)),
        "excluded_rows": int(len(exclusions)),
        "origin_fallback_used": False,
        "max_metric_abs_diff": float(
            comparison[["dir_abs_diff", "fpir_abs_diff"]].max().max()
        ),
    }
result


## 다음 단계

이 노트북은 결과를 변경하지 않습니다. 검증이 완료되면
`01_evaluation_and_visualization.ipynb`에서 동일한 checksum 고정 결과를
읽어 PCA/PQ 프로파일별 DIR·FPIR·저장량 관계를 확인합니다.
